In [4]:
"""
Daily Embedded Option Pricer — CRR Binomial Tree
--------------------------------------------------
For each convertible bond, prices the embedded conversion option
on every trading day using a Cox-Ross-Rubinstein (CRR) binomial tree.

Why CRR Binomial Tree and not Black-Scholes:
    - The conversion option is American-style (can be exercised any day)
    - Black-Scholes only prices European options correctly
    - CRR tree checks for early exercise at every single node
    - Same model will be used in the full bond floor calculation later

Inputs per bond per day:
    S     = stock closing price
    K     = conversion price (fixed, from static data)
    T     = remaining maturity in years (shrinks daily)
    r     = risk-free rate (interpolated from yield curve, zero coupon)
    sigma = 12-month ATM implied volatility (from equity data)

Output:
    data/clean/option_values/
        {TICKER}_options.csv
            date, stock_price, implied_vol, risk_free_rate,
            remaining_maturity, option_value, delta, option_value_per_bond

    data/clean/option_values_all.csv
        All bonds combined in one file
"""

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

# ── Detect which machine we are on ────────────────────────────────────────
import getpass
USER = getpass.getuser()

if USER == "sarda":
    BASE_DIR = r"C:\Users\sarda\Desktop\cba2"
elif USER == "jinay":
    BASE_DIR = r"C:\Users\jinay\Desktop\cba2"
else:
    # Fallback — change this if neither matches
    BASE_DIR = r"C:\Users\sarda\Desktop\cba2"

STATIC_PATH  = os.path.join(BASE_DIR, "data", "raw", "bond_static", "CONV_BOND_DATA.xlsx")
BOND_DIR     = os.path.join(BASE_DIR, "data", "raw", "bond_dynamic")
EQUITY_DIR   = os.path.join(BASE_DIR, "data", "raw", "stock")
DISCOUNT_PATH= os.path.join(BASE_DIR, "data", "clean", "final_discount_rates_daily.csv")
OUTPUT_DIR   = os.path.join(BASE_DIR, "data", "clean", "option_values")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Binomial tree parameters ───────────────────────────────────────────────
N_STEPS = 100    # 100 steps — good balance of accuracy vs speed
                 # increase to 200 for final runs if needed

# Soft call provision: most convertible bonds can be called by the issuer
# if the stock exceeds ~130% of the conversion price for 20/30 consecutive
# trading days. We approximate this in the CRR tree by capping option value
# at intrinsic at nodes where stock exceeds the trigger — no time value
# survives past the call boundary because the issuer terminates the option.
# Set to None to disable (prices to maturity, ignoring calls).
SOFT_CALL_TRIGGER_PCT = None  # Set to 130 to enable soft call cap; None = disabled

# Tickers excluded from pricing — insufficient data or removed from universe
EXCLUDE_TICKERS = {"F", "HASI", "CSWC"}

# ══════════════════════════════════════════════════════════════════════════
# SECTION 1 — CRR BINOMIAL TREE
# ══════════════════════════════════════════════════════════════════════════

def crr_american_call(S, K, T, r, sigma, N=100, call_trigger=None):
    # Edge cases
    if T <= 0:
        return max(S - K, 0.0), 1.0 if S > K else 0.0
    if sigma <= 0:
        return max(S - K, 0.0), 1.0 if S > K else 0.0

    dt   = T / N
    u    = np.exp(sigma * np.sqrt(dt))
    d    = 1 / u
    q    = (np.exp(r * dt) - d) / (u - d)
    disc = np.exp(-r * dt)
    q    = np.clip(q, 0.0, 1.0)

    # Terminal stock prices and option payoffs
    j  = np.arange(N + 1)
    ST = S * (u ** j) * (d ** (N - j))
    V  = np.maximum(ST - K, 0.0)

    # Roll backwards through the tree
    # At step i=1 we save the two node values — that is all we need for delta
    V_step1 = None

    for i in range(N - 1, -1, -1):
        j         = np.arange(i + 1)
        Si        = S * (u ** j) * (d ** (i - j))
        hold      = disc * (q * V[1:i+2] + (1 - q) * V[0:i+1])
        intrinsic = np.maximum(Si - K, 0.0)
        V         = np.maximum(hold, intrinsic)

        # Soft call: if stock exceeds trigger, issuer calls the bond.
        # Bondholder converts → option value capped at intrinsic (no time value).
        if call_trigger is not None:
            V = np.where(Si > call_trigger, intrinsic, V)

        # Save the two nodes at step 1 for delta
        if i == 1:
            V_step1 = V.copy()

    option_value = float(V[0])

    # Delta = (V_up - V_down) / (S_up - S_down) at step 1
    # This is the standard binomial tree delta — no separate tree needed
    if V_step1 is not None and len(V_step1) == 2:
        S_up  = S * u
        S_down = S * d
        delta = (V_step1[1] - V_step1[0]) / (S_up - S_down)
        delta = float(np.clip(delta, 0.0, 1.0))
    else:
        delta = 1.0 if S > K else 0.0

    return option_value, delta


# ══════════════════════════════════════════════════════════════════════════
# SECTION 2 — LOAD DISCOUNT RATES
# ══════════════════════════════════════════════════════════════════════════

def load_discount_table():
    """Load the final_discount_rates_daily.csv built by yield_curve.py"""
    df = pd.read_csv(DISCOUNT_PATH)
    df["Date"] = pd.to_datetime(df["Date"])
    print(f"Discount table loaded: {len(df)} rows | "
          f"{df['Date'].min().date()} to {df['Date'].max().date()}")
    return df


def get_risk_free_rate(discount_df, date, maturity_years):
    """
    Interpolate the risk-free rate (ZC Treasury, no credit spread)
    for a given date and remaining maturity.
    """
    TENORS = [1, 2, 3, 5, 7, 10]

    date = pd.to_datetime(date)
    row  = discount_df[discount_df["Date"] == date]

    if row.empty:
        idx = (discount_df["Date"] - date).abs().idxmin()
        row = discount_df.iloc[[idx]]

    rates = {t: row[f"risk_free_{t}y"].values[0] for t in TENORS}

    if maturity_years <= TENORS[0]:
        return rates[TENORS[0]]
    if maturity_years >= TENORS[-1]:
        return rates[TENORS[-1]]

    lower = max(t for t in TENORS if t <= maturity_years)
    upper = min(t for t in TENORS if t >= maturity_years)

    if lower == upper:
        return rates[lower]

    return rates[lower] + (
        (maturity_years - lower) / (upper - lower)
    ) * (rates[upper] - rates[lower])


# ══════════════════════════════════════════════════════════════════════════
# SECTION 3 — LOAD BOND + EQUITY DATA
# ══════════════════════════════════════════════════════════════════════════

def load_static():
    """Load full static bond file from sheet 'Sheet1'. Ensures 'Ticker' column exists (from 'Ticker' or 'Cmn Stk Tkr')."""
    df = pd.read_excel(STATIC_PATH, sheet_name="Sheet1")
    if "Ticker" not in df.columns or df["Ticker"].isna().all():
        if "Cmn Stk Tkr" in df.columns:
            df = df.copy()
            df["Ticker"] = df["Cmn Stk Tkr"]
    print(f"Static file loaded: {len(df)} bonds")
    return df


def load_equity(ticker):
    """Load equity file for one ticker."""
    path = os.path.join(EQUITY_DIR, f"{ticker}_equity.xlsx")

    if not os.path.exists(path):
        raise FileNotFoundError(f"Equity file not found: {path}")

    df = pd.read_excel(path, skiprows=6)
    df = df.rename(columns={
        "Date"                      : "date",
        "PX_OFFICIAL_CLOSE"         : "stock_price",
        "12MTH_IMPVOL_100.0%MNY_DF" : "implied_vol"
    })
    df["date"] = pd.to_datetime(df["date"])
    df = df[["date", "stock_price", "implied_vol"]].dropna(subset=["date"])
    df["stock_price"] = pd.to_numeric(df["stock_price"], errors="coerce")
    df["implied_vol"]  = pd.to_numeric(df["implied_vol"],  errors="coerce")
    df = df.sort_values("date").reset_index(drop=True)

    return df


# ══════════════════════════════════════════════════════════════════════════
# SECTION 4 — PRICE ONE BOND
# ══════════════════════════════════════════════════════════════════════════

def price_bond_options(ticker, static_row, equity_df, discount_df):
    """
    For one bond, price the embedded conversion option on every trading day.

    Returns a dataframe with one row per date containing:
        date, stock_price, implied_vol, risk_free_rate,
        remaining_maturity, option_value, delta,
        option_value_per_bond (= option_value × conversion_ratio)
    """
    # ── Extract static parameters ──────────────────────────────────────────
    maturity_date     = pd.to_datetime(static_row["Maturity"].values[0])
    conversion_price  = float(static_row["CV Conversion Price"].values[0])
    conversion_ratio  = float(static_row["Conversion Ratio"].values[0])

    # ── Soft call trigger (stock price level above which issuer calls) ────
    call_trigger = (
        conversion_price * SOFT_CALL_TRIGGER_PCT / 100
        if SOFT_CALL_TRIGGER_PCT else None
    )

    # ── Implied vol: already in % (e.g. 35.08), convert to decimal ────────
    equity_df = equity_df.copy()
    equity_df["implied_vol_dec"] = equity_df["implied_vol"] / 100

    results = []
    skipped = 0

    for _, row in equity_df.iterrows():
        date        = row["date"]
        S           = row["stock_price"]
        sigma       = row["implied_vol_dec"]

        # Remaining maturity in years
        T = (maturity_date - date).days / 365.25

        # Skip if already matured or data is missing
        if T <= 0:
            skipped += 1
            continue
        if pd.isna(S) or pd.isna(sigma) or S <= 0 or sigma <= 0:
            skipped += 1
            continue

        # Risk-free rate for this date and maturity
        r = get_risk_free_rate(discount_df, date, T)

        # Price the American call (with soft call provision if enabled)
        option_val, delta = crr_american_call(
            S=S,
            K=conversion_price,
            T=T,
            r=r,
            sigma=sigma,
            N=N_STEPS,
            call_trigger=call_trigger
        )

        # Scale to per-bond value
        # option_value_per_bond = option value per share × conversion ratio
        # This is the dollar value of the option component in one $1,000 bond
        option_per_bond  = option_val * conversion_ratio
        option_value_pct = option_per_bond / 1000 * 100  # % of par

        results.append({
            "date"               : date,
            "stock_price"        : round(S, 4),
            "conversion_price"   : round(conversion_price, 4),
            "implied_vol_pct"    : round(row["implied_vol"], 4),
            "risk_free_rate_pct" : round(r * 100, 4),
            "remaining_maturity" : round(T, 4),
            "option_value"       : round(option_val, 6),
            "delta"              : round(delta, 6),
            "option_value_per_bond": round(option_per_bond, 4),
            "option_value_pct"   : round(option_value_pct, 6),
        })

    df_out = pd.DataFrame(results)

    if skipped > 0:
        print(f"    Skipped {skipped} rows (matured / missing data)")

    return df_out


# ══════════════════════════════════════════════════════════════════════════
# SECTION 5 — RUN ALL BONDS
# ══════════════════════════════════════════════════════════════════════════

def run_all_bonds():
    """
    Loop through every bond in the static file,
    price options daily, save individual CSVs and one combined file.
    """
    print("=" * 65)
    print("  DAILY OPTION PRICING — CRR BINOMIAL TREE (American Call)")
    print(f"  Steps: {N_STEPS}")
    if SOFT_CALL_TRIGGER_PCT:
        print(f"  Soft call trigger: {SOFT_CALL_TRIGGER_PCT}% of conversion price")
    else:
        print("  Soft call: DISABLED (pricing to maturity)")
    print("=" * 65)

    # Load shared data (load_static normalizes Ticker from Cmn Stk Tkr if needed)
    static_df   = load_static()
    discount_df = load_discount_table()

    all_results = []
    tickers = [
        t for t in static_df["Ticker"].dropna().unique().tolist()
        if t not in EXCLUDE_TICKERS
    ]

    if not tickers:
        sample = static_df["Ticker"].head(5).tolist()
        print(f"\nFound 0 bonds to price. Ticker column sample: {sample}")
        print("  Check CONV_BOND_DATA.xlsx: ensure column 'Ticker' or 'Cmn Stk Tkr' has ticker symbols.")
        return []

    print(f"\nFound {len(tickers)} bonds to price: {tickers}\n")

    for i, ticker in enumerate(tickers, 1):
        print(f"[{i}/{len(tickers)}] Pricing {ticker}...")

        try:
            # Get static row
            static_row = static_df[static_df["Ticker"] == ticker]

            # Load equity data
            equity_df = load_equity(ticker)

            # Price options daily
            result_df = price_bond_options(
                ticker, static_row, equity_df, discount_df
            )

            if result_df.empty:
                print(f"    WARNING: No results produced for {ticker}")
                continue

            result_df.insert(0, "ticker", ticker)

            # Save individual file
            out_path = os.path.join(OUTPUT_DIR, f"{ticker}_options.csv")
            result_df.to_csv(out_path, index=False)

            # Summary stats
            print(f"    Rows: {len(result_df)} | "
                  f"{result_df['date'].min().date()} to "
                  f"{result_df['date'].max().date()}")
            print(f"    Option value range: "
                  f"${result_df['option_value'].min():.2f} – "
                  f"${result_df['option_value'].max():.2f} per share")
            print(f"    Per bond range:     "
                  f"${result_df['option_value_per_bond'].min():.2f} – "
                  f"${result_df['option_value_per_bond'].max():.2f}")
            print(f"    Delta range:        "
                  f"{result_df['delta'].min():.3f} – "
                  f"{result_df['delta'].max():.3f}")
            print(f"    Saved → option_values/{ticker}_options.csv\n")

            all_results.append(result_df)

        except FileNotFoundError as e:
            print(f"    SKIPPED — file not found: {e}\n")
        except Exception as e:
            print(f"    ERROR — {ticker}: {e}\n")

    # Save combined file
    if all_results:
        combined = pd.concat(all_results, ignore_index=True)
        combined_path = os.path.join(
            BASE_DIR, "data", "clean", "option_values_all.csv"
        )
        combined.to_csv(combined_path, index=False)
        print(f"\nCombined file saved → data/clean/option_values_all.csv")
        print(f"Total rows: {len(combined)} across {len(all_results)} bonds")

    return all_results


# ══════════════════════════════════════════════════════════════════════════
# SECTION 6 — TEST SINGLE BOND
# ══════════════════════════════════════════════════════════════════════════

def test_single_bond(ticker="MTH"):
    """
    Quick test for one bond — prints a sample of results
    and a few spot-check valuations.
    Run this first before running all bonds.
    """
    print("=" * 65)
    print(f"  TEST RUN — {ticker}")
    print("=" * 65)

    static_df   = load_static()
    discount_df = load_discount_table()
    static_row  = static_df[static_df["Ticker"] == ticker]
    equity_df   = load_equity(ticker)

    print(f"\nStatic parameters for {ticker}:")
    print(f"  Maturity:          {static_row['Maturity'].values[0]}")
    print(f"  Conversion Price:  {static_row['CV Conversion Price'].values[0]}")
    print(f"  Conversion Ratio:  {static_row['Conversion Ratio'].values[0]}")

    result_df = price_bond_options(ticker, static_row, equity_df, discount_df)
    result_df.insert(0, "ticker", ticker)

    print(f"\nFirst 5 rows:")
    print(result_df[["date", "stock_price", "conversion_price",
                      "implied_vol_pct", "risk_free_rate_pct",
                      "remaining_maturity", "option_value",
                      "delta", "option_value_per_bond"]].head().to_string())

    print(f"\nLast 5 rows:")
    print(result_df[["date", "stock_price", "conversion_price",
                      "implied_vol_pct", "risk_free_rate_pct",
                      "remaining_maturity", "option_value",
                      "delta", "option_value_per_bond"]].tail().to_string())

    print(f"\nSummary statistics:")
    print(result_df[["option_value", "delta",
                      "option_value_per_bond"]].describe().round(4))

    # Bloomberg comparison check
    bbg_delta  = static_row["CV Model Delta S %"].values[0] / 100
    bbg_option = static_row["CV Equity Option Value"].values[0]

    print(f"\nBloomberg benchmark comparison (most recent date):")
    latest = result_df.iloc[-1]
    print(f"  {'':30} {'Bloomberg':>12} {'Our Model':>12}")
    print(f"  {'-'*55}")
    print(f"  {'Delta':30} {bbg_delta:>12.4f} {latest['delta']:>12.4f}")
    print(f"  {'Option Value (per share)':30} {bbg_option:>12.4f} "
          f"{latest['option_value']:>12.4f}")

    return result_df


# ══════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":

    import sys

    if len(sys.argv) > 1 and sys.argv[1] == "test":
        # Run: python option_pricer.py test
        # Or:  python option_pricer.py test AAPL
        ticker = sys.argv[2] if len(sys.argv) > 2 else "MTH"
        test_single_bond(ticker)

    else:
        # Run: python option_pricer.py
        # Prices all bonds
        run_all_bonds()

  DAILY OPTION PRICING — CRR BINOMIAL TREE (American Call)
  Steps: 100
  Soft call: DISABLED (pricing to maturity)
Static file loaded: 22 bonds
Discount table loaded: 1874 rows | 2019-01-02 to 2026-03-09

Found 19 bonds to price: ['KRG', 'PPL1', 'EXC', 'BXP', 'EEFT', 'WEC1', 'DLR', 'WEC2', 'WEC3', 'REXR1', 'REXR2', 'MTH', 'NEE', 'FRT', 'EVRG', 'CDP', 'DUK', 'VTR', 'PPL2']

[1/19] Pricing KRG...
    Skipped 60 rows (matured / missing data)
    Rows: 1184 | 2021-03-22 to 2026-03-04
    Option value range: $1.73 – $11.41 per share
    Per bond range:     $73.21 – $483.40
    Delta range:        0.507 – 0.809
    Saved → option_values/KRG_options.csv

[2/19] Pricing PPL1...
    Rows: 68 | 2025-11-24 to 2026-03-04
    Option value range: $4.38 – $8.20 per share
    Per bond range:     $102.59 – $192.10
    Delta range:        0.522 – 0.653
    Saved → option_values/PPL1_options.csv

[3/19] Pricing EXC...
    Rows: 61 | 2025-12-04 to 2026-03-04
    Option value range: $3.19 – $6.27 per shar